In [2]:
from glob import glob
import pandas as pd
DATA_DIR = "/DATA/disk2/yuhang/.cache/modelscope/datasets/deepctrl/deepctrl-sft-data/sft_data_en.jsonl"
from tqdm import tqdm
import json
import re
import random


In [3]:
# 读取数据集并统计type_keyword
# 注意：type_keyword是一个列表，需要对列表中的每个关键词进行统计
type_keyword_count = {}  # 用于统计每个type_keyword的出现次数

print("开始读取数据集并统计type_keyword...")

with open(DATA_DIR, 'r', encoding='utf-8') as file:
    for idx, line in enumerate(tqdm(file, desc="处理数据")):
        try:
            # 解析JSON数据
            json_obj = json.loads(line)
            
            # 提取type_keyword字段
            keyword_list = json_obj.get("type_keyword", "")
            
            for keyword in keyword_list:
            # 如果type_keyword不为空，则进行统计
                if keyword in type_keyword_count:
                    type_keyword_count[keyword] += 1
                else:
                    type_keyword_count[keyword] = 1
        except json.JSONDecodeError:
            # 如果某行JSON格式有问题，跳过该行
            print(f"第{idx+1}行JSON格式错误，跳过")
            continue

# 显示统计结果
print(f"\n总共找到 {len(type_keyword_count)} 种不同的type_keyword")
print("\ntype_keyword统计结果:")
print("-" * 50)

# 按出现次数降序排列显示结果
sorted_keywords = sorted(type_keyword_count.items(), key=lambda x: x[1], reverse=True)
for keyword, count in sorted_keywords:
    print(f"{keyword}: {count} 次")

print(f"\n总数据条数: {sum(type_keyword_count.values())}")


开始读取数据集并统计type_keyword...


处理数据: 2767403it [00:37, 73351.49it/s] 


总共找到 255 种不同的type_keyword

type_keyword统计结果:
--------------------------------------------------
write: 917521 次
code: 728986 次
provid: 706650 次
data: 592985 次
creat: 579295 次
descript: 497410 次
base: 460418 次
includ: 399027 次
make: 396619 次
return: 388240 次
notebook: 380695 次
jupyt: 380695 次
year: 375459 次
question: 351856 次
function: 324434 次
work: 307494 次
impact: 306737 次
patient: 294274 次
medic: 294274 次
pleas: 272404 次
valu: 266795 次
like: 265184 次
might: 259617 次
design: 249188 次
feel: 246733 次
time: 246539 次
http: 242668 次
print: 223281 次
doctor: 222151 次
answer: 222151 次
pain: 222151 次
gener: 213952 次
model: 213685 次
import: 211805 次
help: 208120 次
input: 201016 次
string: 200251 次
develop: 198893 次
first: 187353 次
error: 185927 次
follow: 184429 次
would: 182818 次
file: 171867 次
request: 169182 次
person: 165210 次
public: 164959 次
product: 163176 次
differ: 159433 次
market: 158459 次
also: 158025 次
histori: 157098 次
financi: 153552 次
text: 149229 次
social: 148888 次
number: 146539 次

In [4]:
def convert_to_llama_factory_format(item):
    """
    将SFT数据格式转换为llama-factory格式
    原格式：instruction为空，input包含任务描述，output包含生成内容
    转换后：instruction包含任务描述，input为空，output保持不变
    """
    return {
        "instruction": item["input"],
        "input": "",
        "output": item["output"].strip()
    }

all_data = []
token_num_list = []

with open(DATA_DIR, 'r', encoding='utf-8') as file:
    for idx, line in enumerate(file):
        # 解码JSON对象
        json_obj = json.loads(line)
        
        # 检查必要字段是否存在且有效
        if json_obj.get("input", "") == "" or json_obj.get("output", "") == "":
            continue
            
        # 转换为llama-factory格式
        converted_data = convert_to_llama_factory_format(json_obj)
        all_data.append(converted_data)
        token_num_list.append(len(converted_data))

print(f"总共处理了 {len(all_data)} 条数据")
print(len(token_num_list))

# 显示转换后的数据示例
if all_data:
    print("转换后的数据示例:")
    print(all_data[0])

总共处理了 2767075 条数据
2767075
转换后的数据示例:
{'instruction': 'Extract the main points from the essay.', 'input': '', 'output': 'The essay explores the impact of technology on human communication. It states that while technology has made communication more convenient, it has also led to a decrease in the quality of human interaction. The abundance of online communication has led to a decrease in social skills and the ability to read non-verbal cues. The essay concludes that it is important to not forget the value of face-to-face communication.'}


In [6]:
import numpy as np
np.percentile(token_num_list,[50, 60, 70, 80, 90,95,99])
print(np.percentile(token_num_list,[50, 60, 70, 80, 90,95, 99]))
filtered_long_data = []
filtered_token_num = []
for i in range(len(all_data)):
    if token_num_list[i] < 768:
        filtered_long_data.append(all_data[i])
        filtered_token_num.append(token_num_list[i])
print(np.percentile(filtered_token_num,[50, 60, 70, 80, 90,95, 99]))
print(len(filtered_token_num))

[3. 3. 3. 3. 3. 3. 3.]
[3. 3. 3. 3. 3. 3. 3.]
2767075


In [7]:
few_data = random.sample(filtered_long_data, len(filtered_long_data))
print(f"采样后数据量: {len(few_data)}")

采样后数据量: 2767075


In [8]:
import os
file_path = "/DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input/deepctl_276W_en.jsonl"

directory = os.path.dirname(file_path)

if not os.path.exists(directory):
    os.makedirs(directory)
    print(f"目录{directory} 不存在，已创建")
else:
    print(f"目录{directory} 已存在")

with open(file_path, 'w', encoding="utf-8") as f:
    for item in few_data:
        json.dump(item, f, ensure_ascii=False)
        f.write('\n')

print(f"数据已保存到: {file_path}")

目录/DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input 已存在
数据已保存到: /DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input/deepctl_276W_en.jsonl
